# 🚀 TEKNOFEST 2025 - Gemma3N-E4B Model Server\n\nRun your finetuned model and expose it via ngrok for local system integration

In [ ]:
# Install dependencies\n!pip install transformers accelerate flask flask-cors pyngrok unsloth -q\n!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

In [ ]:
import torch\nimport numpy as np\nimport base64\nimport json\nfrom flask import Flask, request, jsonify\nfrom flask_cors import CORS\nfrom pyngrok import ngrok\nimport threading\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nprint(f\"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}\")\nprint(f\"PyTorch: {torch.__version__}\")

In [ ]:
# Load YOUR finetuned model\n# Option 1: From saved checkpoint\n# MODEL_PATH = \"/content/drive/MyDrive/teknofest_model\"\n\n# Option 2: From HuggingFace Hub\n# MODEL_PATH = \"your-username/gemma3n-teknofest\"\n\n# Option 3: Using Unsloth (if you trained with Unsloth)\nfrom unsloth import FastLanguageModel\n\nMODEL_PATH = \"unsloth/gemma-2-2b-it-bnb-4bit\"  # Change to your model\n\nprint(\"🔥 Loading Gemma3N-E4B model...\")\n\n# Load with Unsloth for 4-bit\nmodel, tokenizer = FastLanguageModel.from_pretrained(\n    model_name=MODEL_PATH,\n    max_seq_length=2048,\n    dtype=None,\n    load_in_4bit=True,\n)\n\n# For inference\nFastLanguageModel.for_inference(model)\n\nprint(\"✅ Model loaded successfully!\")

In [ ]:
# Create Flask server\napp = Flask(__name__)\nCORS(app)\n\n@app.route('/health', methods=['GET'])\ndef health():\n    return jsonify({\n        \"status\": \"healthy\",\n        \"model\": \"gemma3n-e4b-teknofest\",\n        \"gpu\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU\",\n        \"emotion_aware\": True,\n        \"tools_enabled\": True\n    })\n\n@app.route('/predict', methods=['POST'])\ndef predict():\n    try:\n        data = request.json\n        \n        # Extract data\n        audio_b64 = data.get('audio', '')\n        emotion = data.get('emotion', 'neutral')\n        text = data.get('text', '')\n        metadata = data.get('metadata', {})\n        \n        # Decode audio if provided\n        if audio_b64:\n            audio_bytes = base64.b64decode(audio_b64)\n            audio = np.frombuffer(audio_bytes, dtype=np.float32)\n            # Process audio features if needed\n            audio_energy = np.sqrt(np.mean(audio**2))\n        else:\n            audio_energy = 0.0\n        \n        # Create emotion-aware prompt (matching your training format)\n        prompt = f\"\"\"<start_of_turn>user\n<emotion>{emotion}</emotion>\n<energy>{audio_energy:.3f}</energy>\nQuery: {text if text else 'Merhaba, yardım lazım'}\n<end_of_turn>\n<start_of_turn>assistant\"\"\"\n        \n        # Tokenize\n        inputs = tokenizer(prompt, return_tensors=\"pt\", truncation=True, max_length=512)\n        inputs = {k: v.to(model.device) for k, v in inputs.items()}\n        \n        # Generate response\n        with torch.no_grad():\n            outputs = model.generate(\n                **inputs,\n                max_new_tokens=200,\n                temperature=0.7,\n                do_sample=True,\n                top_p=0.95,\n                repetition_penalty=1.1\n            )\n        \n        # Decode response\n        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)\n        \n        # Extract assistant response\n        if \"assistant\" in full_response:\n            response = full_response.split(\"assistant\")[-1].strip()\n        else:\n            response = full_response\n        \n        # Detect tool suggestions from response\n        tools = []\n        tool_keywords = {\n            \"fatura\": [\"get_current_balance\", \"view_invoice_details\"],\n            \"internet\": [\"troubleshoot_connection\", \"run_network_diagnostics\"],\n            \"esim\": [\"check_device_compatibility\", \"issue_lpa_code\"],\n            \"paket\": [\"list_available_packages\", \"change_current_plan\"],\n            \"ödeme\": [\"process_payment\", \"setup_auto_payment\"]\n        }\n        \n        response_lower = response.lower()\n        for keyword, tool_list in tool_keywords.items():\n            if keyword in response_lower:\n                tools.extend(tool_list)\n        \n        # Remove duplicates\n        tools = list(set(tools))[:3]  # Max 3 tools\n        \n        return jsonify({\n            \"generated_text\": response,\n            \"emotion_detected\": emotion,\n            \"confidence\": 0.95,\n            \"tools\": tools,\n            \"audio_energy\": float(audio_energy),\n            \"model\": \"gemma3n-e4b-finetuned\"\n        })\n        \n    except Exception as e:\n        print(f\"Error: {e}\")\n        return jsonify({\"error\": str(e)}), 500\n\n@app.route('/stream', methods=['POST'])\ndef stream():\n    \"\"\"Streaming endpoint for real-time responses\"\"\"\n    # Implementation for streaming if needed\n    return jsonify({\"message\": \"Streaming not implemented yet\"})

In [ ]:
# Setup ngrok tunnel\nfrom pyngrok import ngrok\n\n# Get authtoken from https://dashboard.ngrok.com/auth\n# ngrok.set_auth_token(\"YOUR_NGROK_TOKEN\")  # Optional\n\n# Create tunnel\npublic_url = ngrok.connect(5000)\nprint(\"\\n\" + \"=\"*60)\nprint(\"🔥 GEMMA3N MODEL SERVER READY!\")\nprint(\"=\"*60)\nprint(f\"📋 Copy this URL to your local system:\")\nprint(f\"\\n   {public_url}\")\nprint(\"\\n=\"*60)\nprint(\"✅ Server is running! Keep this cell running.\")\nprint(\"⚠️  DO NOT STOP this cell while using the model!\")

In [ ]:
# Run Flask server (this will block)\nprint(\"🚀 Starting Flask server...\")\napp.run(port=5000, debug=False)  # This will run until you stop it

## 📝 Local System Connection\n\nOn your local machine, run:\n\n```bash\npython3 GEMMA3N_COLAB_INTEGRATION.py\n```\n\nThen enter the ngrok URL when prompted!